<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/4_model_training/4_10_model_tft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4_10_model_tft

TFT: Temporal Fusion Transformer

## Introducción y Resumen

Temporal Fusion Transformer (TFT) fue propuesto por Google (Lim et al., 2020).
Está diseñado específicamente para series temporales multivariadas y combina lo mejor de varios mundos:

- LSTM → para capturar dependencias temporales locales.

- Self-Attention (Transformer) → para capturar relaciones de largo plazo y entre features.

- Gating + Variable Selection Networks → para seleccionar dinámicamente qué features son más relevantes en cada instante.

- Interpretabilidad → puedes visualizar la importancia temporal y por variable.

👉 En tu caso:

- Tienes ventanas fijas de 60 minutos (window_size=60).
- Cada ventana tiene muchas features (entre 900 y 1080), con relaciones complejas.
- Necesitas capturar patrones secuenciales y relevancia entre indicadores técnicos y alpha factors.

➡️ El TFT es ideal.


## 0. Configuración del Entorno


### 0.1. Instalación de librerías


### 0.2. Importación de librerías


In [4]:
# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
import sklearn, scipy, numpy #, optuna

import joblib

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

import sys, platform, lightgbm as lgb
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [5]:
print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
#print("optuna:", optuna.__version__)
#print("xgboost:", xgb.__version__)
#print("lightgbm:", lgb.__version__)

# CatBoost usa la clase para exponer versión
#print("catboost:", catboost.__version__)

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
numpy: 2.0.2
scipy: 1.16.3
sklearn: 1.6.1


### 0.3. Acceso a Drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [7]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [8]:
#mnq_train = load_data("train")
#mnq_valid = load_data("valid")
#mnq_test = load_data("test")

### 1.2. Información de datasets


In [9]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [10]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [11]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_30 = features_dict["features_to_30"]
features_to_60 = features_dict["features_to_60"]
features_to_90 = features_dict["features_to_90"]


In [12]:
print(f'Listado de features para 30min ({len(features_to_30)}): {features_to_30}')
print(f'Listado de features para 60min ({len(features_to_60)}): {features_to_60}')
print(f'Listado de features para 90min ({len(features_to_90)}): {features_to_90}')

Listado de features para 30min (5): ['ire_90', 'rev_mom_z_90', 'roc_60', 'rev_score_90', 'price_ema30']
Listado de features para 60min (7): ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'rev_mom_vol_z_60', 'momentum_5', 'roc_20']
Listado de features para 90min (7): ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

### 2.0. Funciones

#### Función para cargar ventanas

In [13]:
def load_windows_and_scaler(target: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz'
    path_valid  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz'
    path_test   = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/3_dataset_preparation/global_scaler_{target}.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### Función para revisar información de ventanas

In [14]:
def xy_info(target: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información para horizonte de {target} minutos:')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{target})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{target})')

        print(f'\t{y.shape[0]} targets.')
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

    return  X_train.shape[0], X_valid.shape[0], X_test.shape[0]

In [15]:
features_base = ['open', 'high', 'low', 'close', 'volume']

features_30 = features_base + features_to_30
features_60 = features_base + features_to_60
features_90 = features_base + features_to_90

window_size = 90

### 2.1 Carga de ventanas 30 minutos

In [16]:
X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30, scaler_30 = load_windows_and_scaler(target = '30')

In [17]:
n_samples_train_30, n_samples_valid_30, n_samples_test_30 = xy_info( '30', X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30)

Información para horizonte de 30 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	193487 targets.
	Distribución y: mean=0.000056, std=0.002760, min=-0.028931, max=0.032372

Set de validación:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=0.000103, std=0.003273, min=-0.023167, max=0.068056

Set de testeo:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=-0.000023, std=0.002407, min=-0.015982, max=0.016097


### 2.2 Carga de ventanas 60 minutos

In [18]:
X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60, scaler_60 = load_windows_and_scaler(target = '60')

In [19]:
n_samples_train_60, n_samples_valid_60, n_samples_test_60 = xy_info( '60', X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60)

Información para horizonte de 60 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	193487 targets.
	Distribución y: mean=0.000114, std=0.003907, min=-0.040179, max=0.036224

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=0.000163, std=0.004643, min=-0.038084, max=0.079896

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=-0.000068, std=0.003521, min=-0.018494, max=0.020365


### 2.3 Carga de ventanas 90 minutos

In [20]:
X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90, scaler_90 = load_windows_and_scaler(target = '90')

In [21]:
n_samples_train_90, n_samples_valid_90, n_samples_test_90 = xy_info('90', X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90)

Información para horizonte de 90 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	193487 targets.
	Distribución y: mean=0.000159, std=0.004746, min=-0.037742, max=0.039284

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=0.000224, std=0.005887, min=-0.040748, max=0.083184

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=-0.000132, std=0.004558, min=-0.023392, max=0.026213


## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [22]:
def load_metrics(data: str):
    data_path = f'{drive_path}/4_model_training/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [23]:
def metrics_verify(data: str) -> bool:
    data_path = f"{drive_path}/4_model_training/{data}.parquet"
    return os.path.exists(data_path)


In [24]:
def load_or_create_metrics (data:str):
  if metrics_verify(data):
      print(f"Las métricas existen y son almacenadas en {data[4:len(data)]}")
      model_metrics = load_metrics(data)
      #print(random_forest_metrics)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[4:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [25]:
tft_metrics, metrics = load_or_create_metrics("4_10_tft_metrics")

Las métricas no existen. Se crea el dataset _tft_metrics para almacenar las métricas


### 3.2. Función para guardar métricas

In [26]:
def save_metrics (metrics,  metrics_name: str):
  metrics_path = f"{drive_path}/4_model_training/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [27]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [28]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

## 4. Re-formateo más Encoder mínimo

Helper para re-formatear nuestras ventanas 2D a 3D que el formato que el modelo necesita.

### 4.1. Helper: de 2D (aplanado) a 3D (B, T, F)

Lo usamos para cada set y horizonte. Nuestro `window_size = 90` y los `n_features` depende del horizonte de tiempo. El siguiente código valida que `windows_size * n_features == X.shape[1]`

In [29]:
import numpy as np
import torch
import torch.nn as nn
import math

def reshape_windows(X_flat: np.ndarray, window_size: int, n_features: int) -> np.ndarray:
    """
    Convierte X de (N, window_size * n_features) a (N, window_size, n_features).
    Valida la consistencia del producto.
    """
    assert X_flat.ndim == 2, "Se esperaba X_flat con 2D (N, T*F)."
    N, TF = X_flat.shape
    assert window_size * n_features == TF, (
        f"Inconsistencia: {window_size} * {n_features} != {TF}"
    )
    return X_flat.reshape(N, window_size, n_features)

In [30]:
window_size = 90
features_base = ['open','high','close','low','volume']

In [31]:
n_features_30 = len (features_30)
n_features_60 = len (features_60)
n_features_90 = len (features_90)

In [32]:
# Re-shape de tus matrices 2D -> 3D
Xtr_30   =   reshape_windows(X_train_30_scaled, window_size, n_features_30)
Xva_30  =   reshape_windows(X_valid_30_scaled, window_size, n_features_30)
Xte_30  = reshape_windows(X_test_30_scaled, window_size, n_features_30)

Xtr_60   =   reshape_windows(X_train_60_scaled, window_size, n_features_60)
Xva_60  =   reshape_windows(X_valid_60_scaled, window_size, n_features_60)
Xte_60  = reshape_windows(X_test_60_scaled, window_size, n_features_60)

Xtr_90   =   reshape_windows(X_train_90_scaled, window_size, n_features_90)
Xva_90  =   reshape_windows(X_valid_90_scaled, window_size, n_features_90)
Xte_90  = reshape_windows(X_test_90_scaled, window_size, n_features_90)

# Comprobación de shapes

print('\nReshape de ventanas 30min:\n')
print(f'\tXtr_30.shape:\t{Xtr_30.shape}')
print(f'\tXva_30.shape:\t{Xva_30.shape}')
print(f'\tXte_30.shape:\t{Xte_30.shape}')

print('\nReshape de ventanas 60min:\n')
print(f'\tXtr_60.shape:\t{Xtr_60.shape}')
print(f'\tXva_60.shape:\t{Xva_60.shape}')
print(f'\tXte_60.shape:\t{Xte_60.shape}')

print('\nReshape de ventanas 90min:\n')
print(f'\tXtr_90.shape:\t{Xtr_90.shape}')
print(f'\tXva_90.shape:\t{Xva_90.shape}')
print(f'\tXte_90.shape:\t{Xte_90.shape}')



Reshape de ventanas 30min:

	Xtr_30.shape:	(193487, 90, 10)
	Xva_30.shape:	(41567, 90, 10)
	Xte_30.shape:	(41567, 90, 10)

Reshape de ventanas 60min:

	Xtr_60.shape:	(193487, 90, 12)
	Xva_60.shape:	(41567, 90, 12)
	Xte_60.shape:	(41567, 90, 12)

Reshape de ventanas 90min:

	Xtr_90.shape:	(193487, 90, 12)
	Xva_90.shape:	(41567, 90, 12)
	Xte_90.shape:	(41567, 90, 12)


| Horizonte | Shape (n_samples, window_size, n_features) | Total features por ventana |
| --------- | ------------------------------------------ | -------------------------- |
| 30 min    | (193 487, **90**, **10**)                  | 900                        |
| 60 min    | (193 487, **90**, **12**)                  | 1 080                      |
| 90 min    | (193 487, **90**, **12**)                  | 1 080                      |


In [33]:
ytr_30 = y_train_30
yva_30 = y_valid_30
yte_30 = y_test_30

ytr_60 = y_train_60
yva_60 = y_valid_60
yte_60 = y_test_60

ytr_90 = y_train_90
yva_90 = y_valid_90
yte_90 = y_test_90

In [34]:
print(ytr_30.shape)
print(yva_30.shape)
print(yte_90.shape)

(193487,)
(41567,)
(41567,)


## 4. No ejecutras

### 4.2. Convertir a tensores


In [35]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [36]:
def create_dataloader (Xtr, Xva, Xte, ytr, yva, yte):

  Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
  Xva_t = torch.tensor(Xva, dtype=torch.float32)
  Xte_t = torch.tensor(Xte, dtype=torch.float32)

  ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(-1)
  yva_t = torch.tensor(yva, dtype=torch.float32).unsqueeze(-1)
  yte_t = torch.tensor(yte, dtype=torch.float32).unsqueeze(-1)

  # Datasets
  train_ds = TensorDataset(Xtr_t, ytr_t)
  valid_ds = TensorDataset(Xva_t, yva_t)
  test_ds  = TensorDataset(Xte_t, yte_t)

  #Dataloaders
  train_dl = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=4)
  valid_dl = DataLoader(valid_ds, batch_size=256, shuffle=False, num_workers=4)
  test_dl  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=4)

  return train_dl, valid_dl, test_dl


In [37]:
train_30_dl, valid_30_dl, test_30_dl = create_dataloader(Xtr_30, Xva_30, Xte_30, ytr_30, yva_30, yte_30)

In [38]:
train_60_dl, valid_60_dl, test_60_dl = create_dataloader(Xtr_60, Xva_60, Xte_60, ytr_60, yva_60, yte_60)

In [39]:
train_90_dl, valid_90_dl, test_90_dl = create_dataloader(Xtr_90, Xva_90, Xte_90, ytr_90, yva_90, yte_90)

### 4.3. Entrenamiento para validación de pipeline

In [40]:
import torch.nn as nn

class Seq2OneLSTM(nn.Module):
    def __init__(self, n_features, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):                 # x: (B, 90, n_features)
        out, _ = self.lstm(x)             # (B, 90, hidden)
        last = out[:, -1, :]              # (B, hidden)
        return self.head(last)            # (B, 1)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
#model = Seq2OneLSTM(n_features=Xtr.shape[2]).to(device)
#opt = torch.optim.Adam(model.parameters(), lr=1e-3)

model_30 = Seq2OneLSTM(n_features=Xtr_30.shape[2]).to(device)
opt_30 = torch.optim.Adam(model_30.parameters(), lr=1e-3)

model_60 = Seq2OneLSTM(n_features=Xtr_60.shape[2]).to(device)
opt_60 = torch.optim.Adam(model_60.parameters(), lr=1e-3)

model_90 = Seq2OneLSTM(n_features=Xtr_90.shape[2]).to(device)
opt_90 = torch.optim.Adam(model_90.parameters(), lr=1e-3)

loss_fn = nn.MSELoss()

In [41]:
def pipeline_probe(model, loss_fn, opt, train_dl):
    for epoch in range(3):  # corto: solo para verificar
      model.train()
      for xb, yb in train_dl:
          xb, yb = xb.to(device), yb.to(device)
          opt.zero_grad()
          pred = model(xb)
          loss = loss_fn(pred, yb)
          loss.backward()
          opt.step()
    print('Pipeline sin errores, loaders/targets están bien')

In [42]:
pipeline_probe(model_30, loss_fn, opt_30, train_30_dl)

Pipeline sin errores, loaders/targets están bien


In [43]:
pipeline_probe(model_60, loss_fn, opt_60, train_60_dl)

Pipeline sin errores, loaders/targets están bien


In [44]:
pipeline_probe(model_90, loss_fn, opt_90, train_90_dl)

Pipeline sin errores, loaders/targets están bien


## 5. Modelo TFT

El TFT “oficial” (pytorch-forecasting) exige un DataFrame con columnas especiales. Como ya tienes 3D, usa este helper para construirlo (por única vez):

In [35]:
import pandas as pd
import numpy as np

def build_tft_df(X, y):
    # X: (N, T, F) | y: (N,)
    N, T, F = X.shape
    df = pd.DataFrame({
        "time_idx": np.tile(np.arange(T), N),
        "group_id": np.repeat(np.arange(N), T),
        "target":   np.repeat(y, T)
    })
    for i in range(F):
        df[f"feat_{i}"] = X[:,:,i].reshape(-1)
    return df


In [36]:
df_tr_30 = build_tft_df(Xtr_30, ytr_30)
df_va_30 = build_tft_df(Xva_30, yva_30)

In [34]:
df_tr_60 = build_tft_df(Xtr_60, ytr_60)
df_va_60 = build_tft_df(Xva_60, yva_60)

In [3]:
df_tr_90 = build_tft_df(Xtr_90, ytr_90)
df_va_90 = build_tft_df(Xva_90, yva_90)

NameError: name 'Xtr_90' is not defined

## 6. Crear TimeSeriesDataSet:

In [ ]:
# Paquetes compatibles con Python 3.11/3.12
!pip install -q "pytorch-forecasting==1.5.0" "pytorch-lightning==2.4.0" "torch>=2.4,<2.6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 391.5/391.5 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB ? eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 601.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB ? eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 777.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.

In [ ]:
import torch
import pytorch_lightning
import pytorch_forecasting

print("Torch:", torch.__version__)
print("Lightning:", pytorch_lightning.__version__)
print("Forecasting:", pytorch_forecasting.__version__)

In [37]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.models import TemporalFusionTransformer

T = Xtr_30.shape[1]
F = Xtr_30.shape[2]

training_ts = TimeSeriesDataSet(
    df_tr_30,
    time_idx="time_idx",
    target="target",
    group_ids=["group_id"],
    min_encoder_length=T, max_encoder_length=T,
    min_prediction_length=1, max_prediction_length=1,
    time_varying_known_reals=[f"feat_{i}" for i in range(F)],
    time_varying_unknown_reals=["target"],
)

validation_ts = training_ts.from_parameters(training_ts.get_parameters(), df_va_30)

ModuleNotFoundError: No module named 'pytorch_forecasting'

--------------------------------------------------------------------------------------